In [1]:
import pandas as pd

In [8]:
pd.read_csv('../src/mt_rainier_stations.txt', skiprows = 3, sep = '|', header = None)

,0,1,2,3,4,5,6,7
0,CC,ARAT,46.789000,-121.853000,1822.440,Mount Ararat,2020-08-06T00:00:00,2599-12-31T23:59:59
1,CC,CARB,46.988320,-122.005410,872.470,Carbon Repeater,2018-10-16T00:00:00,2599-12-31T23:59:59
2,CC,COPP,46.797980,-121.828750,1895.000,COPP,2022-09-20T11:00:00,2599-12-31T23:59:59
3,CC,CRBN,46.988155,-121.961026,500.080,Carbon River Ranger Station,2020-10-22T00:00:00,2599-12-31T23:59:59
4,CC,CRYS,46.934410,-121.500260,2027.150,Crystal Mountain,2025-09-11T00:00:00,2599-12-31T23:59:59
5,CC,GNOB,46.794140,-121.914380,1659.360,Gobblers Knob Seismic,2022-10-18T13:21:00,2599-12-31T23:59:59
6,CC,GOBB,46.794140,-121.914380,1659.360,Gobbler's Knob,2022-09-07T00:00:00,2599-12-31T23:59:59
7,CC,GTWY,46.740208,-121.916966,617.235,Gateway Entrance Station,2020-10-28T20:00:00,2599-12-31T23:59:59
8,CC,KAUT,46.730263,-121.857381,689.000,Kautz Creek Helibase,2020-09-02T00:00:00,2599-12-31T23:59:59
9,CC,KAVK,46.760410,-122.038560,532.000,Kavik - Ashford Fire Station,2023-08-22T00:00:00,2599-12-31T23:59:59


In [11]:
"""
Fetch seismic channel types (BH/HH/EH/etc.) for each station
from the IRIS FDSN web service.

Requirements: pip install requests pandas obspy
Run: python fetch_station_channels.py
"""

import requests
import pandas as pd
from collections import defaultdict

# ── Station list ────────────────────────────────────────────────────────────
stations = [
    ("CC", "ARAT"), ("CC", "CARB"), ("CC", "COPP"), ("CC", "CRBN"), ("CC", "CRYS"),
    ("CC", "GNOB"), ("CC", "GOBB"), ("CC", "GTWY"), ("CC", "KAUT"), ("CC", "KAVK"),
    ("CC", "LONE"), ("CC", "LONR"), ("CC", "MILD"), ("CC", "OBSR"), ("CC", "OPCH"),
    ("CC", "PANH"), ("CC", "PARA"), ("CC", "PR01"), ("CC", "PR02"), ("CC", "PR03"),
    ("CC", "PR04"), ("CC", "PR05"), ("CC", "RUSH"), ("CC", "SIFT"), ("CC", "TABR"),
    ("CC", "TACK"), ("CC", "TAVI"), ("CC", "VOIT"), ("CC", "WOW"),
    ("UW", "FMW"),  ("UW", "LO2"),  ("UW", "LON"),  ("UW", "RCM"),
    ("UW", "RCS"),  ("UW", "RER"),  ("UW", "STAR"),
]

FDSN_URL = "https://service.iris.edu/fdsnws/station/1/query"

# ── Fetch channel data ───────────────────────────────────────────────────────
records = []
networks = defaultdict(list)
for net, sta in stations:
    networks[net].append(sta)

for net, sta_list in networks.items():
    params = {
        "network": net,
        "station": ",".join(sta_list),
        "level": "channel",
        "format": "text",
        "nodata": "404",
    }
    print(f"Querying network={net} ...")
    resp = requests.get(FDSN_URL, params=params, timeout=30)

    if resp.status_code != 200:
        print(f"  ⚠ HTTP {resp.status_code} for network {net}")
        continue

    for line in resp.text.strip().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        parts = line.split("|")
        if len(parts) < 5:
            continue
        records.append({
            "Network":    parts[0].strip(),
            "Station":    parts[1].strip(),
            "Location":   parts[2].strip(),
            "Channel":    parts[3].strip(),
            "SampleRate": parts[14].strip() if len(parts) > 14 else "",
            "StartTime":  parts[15].strip() if len(parts) > 15 else "",
            "EndTime":    parts[16].strip() if len(parts) > 16 else "",
        })

df_channels = pd.DataFrame(records)
print(f"\nTotal channel records: {len(df_channels)}\n")

# ── Summarise by station ─────────────────────────────────────────────────────
summary = []
for (net, sta), grp in df_channels.groupby(["Network", "Station"]):
    channels      = sorted(grp["Channel"].unique())
    band_types    = sorted(set(ch[:2] for ch in channels))   # e.g. BH, HH, EH
    summary.append({
        "Network":      net,
        "Station":      sta,
        "Band_Types":   ", ".join(band_types),        # compact view
        "All_Channels": ", ".join(channels),           # full list
        "Num_Channels": len(channels),
    })

df_summary = pd.DataFrame(summary)

# ── Print & save ─────────────────────────────────────────────────────────────
print("=== Channel Summary by Station ===")
print(df_summary.to_string(index=False))

df_summary.to_csv("station_channel_summary.csv", index=False)
df_channels.to_csv("station_channels_detail.csv", index=False)
print("\nSaved: station_channel_summary.csv  |  station_channels_detail.csv")

Querying network=CC ...
Querying network=UW ...

Total channel records: 847

=== Channel Summary by Station ===
Network Station                                                                                         Band_Types                                                                                                                                                                                                                                   All_Channels  Num_Channels
     CC    ARAT                                                                                 BD, BH, CD, CH, HD                                                                                                                                                                                                    BDF, BHE, BHN, BHZ, CDF, CHE, CHN, CHZ, HDF             9
     CC    CARB                                                                                             BH, CH                                      

In [12]:
df_summary

,Network,Station,Band_Types,All_Channels,Num_Channels
0,CC,ARAT,"BD, BH, CD, CH, HD","BDF, BHE, BHN, BHZ, CDF, CHE, CHN, CHZ, HDF",9
1,CC,CARB,"BH, CH","BHE, BHN, BHZ, CHE, CHN, CHZ",6
2,CC,COPP,"BD, BH, CD, CH","BDF, BHE, BHN, BHZ, CDF, CHE, CHN, CHZ",8
3,CC,CRBN,"BD, BH, CD, CH","BDF, BHE, BHN, BHZ, CDF, CHE, CHN, CHZ",8
4,CC,CRYS,"CD, CH, HD, HH","CDF, CHE, CHN, CHZ, HDF, HHE, HHN, HHZ",8
5,CC,GNOB,BH,"BHE, BHN, BHZ",3
6,CC,GOBB,BH,"BHE, BHN, BHZ",3
7,CC,GTWY,"BD, BH, CD, CH","BDF, BHE, BHN, BHZ, CDF, CHE, CHN, CHZ",8
8,CC,KAUT,"BD, BH, CD, CH","BDF, BHE, BHN, BHZ, CDF, CHE, CHN, CHZ",8
9,CC,KAVK,BH,"BHE, BHN, BHZ",3


In [13]:
def select_preferred_channels(df_summary):
    """
    For each station, select channels by priority: BH > HH > EH.
    Returns a new dataframe with a 'Selected_Band' and 'Selected_Channels' column.
    """
    PRIORITY = ["BH", "HH", "EH"]
    
    results = []
    for _, row in df_summary.iterrows():
        available_bands = [b.strip() for b in row["Band_Types"].split(",")]
        
        selected = None
        for band in PRIORITY:
            if band in available_bands:
                selected = band
                break
        
        # Filter All_Channels to only the selected band
        if selected:
            selected_channels = [
                ch for ch in row["All_Channels"].split(", ")
                if ch.startswith(selected)
            ]
        else:
            selected_channels = []  # None of BH/HH/EH available

        results.append({
            "Network":           row["Network"],
            "Station":           row["Station"],
            "Available_Bands":   row["Band_Types"],
            "Selected_Band":     selected if selected else "NONE",
            "Selected_Channels": ", ".join(selected_channels),
        })
    
    return pd.DataFrame(results)

df_preferred = select_preferred_channels(df_summary)
print(df_preferred.to_string(index=False))
df_preferred.to_csv("station_preferred_channels.csv", index=False)

Network Station                                                                                    Available_Bands Selected_Band Selected_Channels
     CC    ARAT                                                                                 BD, BH, CD, CH, HD            BH     BHE, BHN, BHZ
     CC    CARB                                                                                             BH, CH            BH     BHE, BHN, BHZ
     CC    COPP                                                                                     BD, BH, CD, CH            BH     BHE, BHN, BHZ
     CC    CRBN                                                                                     BD, BH, CD, CH            BH     BHE, BHN, BHZ
     CC    CRYS                                                                                     CD, CH, HD, HH            HH     HHE, HHN, HHZ
     CC    GNOB                                                                                                 BH    

In [14]:
df_preferred

,Network,Station,Available_Bands,Selected_Band,Selected_Channels
0,CC,ARAT,"BD, BH, CD, CH, HD",BH,"BHE, BHN, BHZ"
1,CC,CARB,"BH, CH",BH,"BHE, BHN, BHZ"
2,CC,COPP,"BD, BH, CD, CH",BH,"BHE, BHN, BHZ"
3,CC,CRBN,"BD, BH, CD, CH",BH,"BHE, BHN, BHZ"
4,CC,CRYS,"CD, CH, HD, HH",HH,"HHE, HHN, HHZ"
5,CC,GNOB,BH,BH,"BHE, BHN, BHZ"
6,CC,GOBB,BH,BH,"BHE, BHN, BHZ"
7,CC,GTWY,"BD, BH, CD, CH",BH,"BHE, BHN, BHZ"
8,CC,KAUT,"BD, BH, CD, CH",BH,"BHE, BHN, BHZ"
9,CC,KAVK,BH,BH,"BHE, BHN, BHZ"


In [15]:
import json

entries = (
    df_preferred[df_preferred["Selected_Band"] != "NONE"]
    [["Network", "Station", "Selected_Band"]]
    .rename(columns={"Network": "net", "Station": "sta", "Selected_Band": "chn"})
    .to_dict(orient="records")
)

print(json.dumps(entries, indent=2))

# Save to file
with open("../src/mt_rainier_stations.json", "w") as f:
    json.dump(entries, f, indent=2)

[
  {
    "net": "CC",
    "sta": "ARAT",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "CARB",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "COPP",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "CRBN",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "CRYS",
    "chn": "HH"
  },
  {
    "net": "CC",
    "sta": "GNOB",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "GOBB",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "GTWY",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "KAUT",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "KAVK",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "LONE",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "LONR",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "MILD",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "OBSR",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "OPCH",
    "chn": "BH"
  },
  {
    "net": "CC",
    "sta": "PANH",
    "chn": "BH"
  },
  {
    "net": "CC",
 